# Analyse des résultats 

In [23]:
import pandas as pd

### Merge des resultats avec dict bbg 

In [24]:
def merge_type(res):
    dict_bbg = pd.read_excel('C:\\Users\\diane\\time-series-pred\\Datasets\\230210 Factor Set - Dictionnary (bbg only).xlsx')
    df_final = pd.merge(res, dict_bbg[['SYMBOL', 'TYPE', 'TYPE2']], on='SYMBOL', how='left')
    cols = ['SYMBOL', 'TYPE', 'TYPE2'] + [c for c in res.columns if c != 'SYMBOL']
    df_final = df_final[cols]
    return df_final

In [25]:
res_ar = pd.read_csv('Resultats/resultats_all_tickers_AR(1).csv')

In [26]:
df_ar = merge_type(res_ar)
df_ar.head()

,SYMBOL,TYPE,TYPE2,MSE,OLS_R2,OLS_Intercept,OLS_Slope,OLS_P_Value_Intercept,P_Value_Slope
0,ADBF Index,Stock Market,Financials,0.000261,0.005777,0.000271,0.497567,0.350072,4.184202e-05
1,ADCM Index,Stock Market,Services,0.000528,0.006266,0.000580,0.388187,0.170883,3.824884e-05
2,ADCT Index,Stock Market,Industrials,0.000304,0.017216,0.000094,0.663209,0.753838,1.307078e-12
3,ADEG Index,Stock Market,Energy,0.000757,0.003857,-0.000040,0.456944,0.932014,1.244449e-03
4,ADHC Index,Stock Market,Health Care,0.000974,0.000828,-0.000178,0.178875,0.897489,5.209936e-01


In [27]:
def stats_par_type(df_final):
    df_final['is_significant'] = df_final['P_Value_Slope'] < 0.05

    stats_by_type = df_final.groupby('TYPE').agg({
        'P_Value_Slope': ['mean', 'std', 'min', 'max'], # Distribution des p-values
        'MSE': ['mean', 'std'],                         # Erreur moyenne et volatilité de l'erreur
        'OLS_R2': ['mean', 'median'],                   # Qualité d'ajustement moyenne
        'OLS_Slope': 'mean',                            # Sensibilité (Beta) moyenne
        'is_significant': 'sum'                         # Nombre d'index significatifs
    }).rename(columns={'sum': 'count_significant'})

    stats_by_type2 = df_final.groupby('TYPE2').agg({
        'MSE': 'mean',
        'OLS_R2': 'mean',
        'OLS_Slope': 'mean',
        'P_Value_Slope': 'median'
    })

    return stats_by_type, stats_by_type2

In [28]:
stats_type_ar, stats_type2_ar = stats_par_type(df_ar)

print("--- Statistiques par Type ---")
print(stats_type_ar)

print("\n--- Statistiques par Type 2 (Secteurs) ---")
print(stats_type2_ar)
print("Ratio de significativité par type :")
stats_type_ar['significance_rate'] = stats_type_ar[('is_significant', 'count_significant')] / df_ar.groupby('TYPE').size()

--- Statistiques par Type ---
                 P_Value_Slope                                          \
                          mean       std            min            max   
TYPE                                                                     
Commodity         2.660308e-01  0.321802   1.354849e-44   9.904855e-01   
Corporate Bonds   1.286786e-01  0.285105  7.880041e-260   8.696342e-01   
Currency          2.776514e-01  0.298280   0.000000e+00   9.537381e-01   
Equity Factors    1.932145e-01  0.283070   1.642318e-38   9.479186e-01   
Hedge Funds       6.740519e-02  0.116566   8.316800e-05   2.020040e-01   
Money Market      8.979813e-02  0.190402   0.000000e+00   8.443086e-01   
Real estate       1.502443e-02  0.045045   1.840678e-48   1.351445e-01   
Sovereign Bonds   1.935879e-01  0.291107   3.269984e-56   9.902809e-01   
Stock Market      2.292072e-01  0.292832  1.605606e-191   9.822514e-01   
Volatility       3.929919e-168  0.000000   0.000000e+00  2.750944e-167   

       